# Модуль 4 · Урок 2 — Якість коду: PEP 8, type hints, docstrings та інструменти

👋 Код пишуть **один раз**, а читають **десятки разів** — колеги й ви самі через півроку. Тому «працює» — це замало; код має бути **читабельним, послідовним і зрозумілим**. Цим і займається культура якості коду. Це прямо впливає на те, чи візьмуть вас у команду.

**Передумова:** Модулі 1–2 (Python, функції, ООП). Урок 1 цього модуля (Git) — інструменти нижче зручно чіпляти до Git через pre-commit.

### Що ви вмітимете після уроку
- писати код за **PEP 8** й розуміти принципи **DRY / KISS / YAGNI**;
- додавати **type hints** (анотації типів) і **docstrings**;
- користуватися інструментами якості: **flake8, black, isort, mypy, ruff**;
- налаштувати **pre-commit hooks**, щоб брудний код не потрапляв у репозиторій.

> 📌 **🔎 Важливо знати** — часто питають на співбесідах. **🚀 Middle+** — глибші деталі.

## 1. Навіщо дбати про якість коду

- 👀 **Читабельність:** «код читають частіше, ніж пишуть» (Гвідо ван Россум). Зрозумілий код = менше багів і швидша розробка.
- 🤝 **Командність:** єдиний стиль → будь-хто одразу читає чужий код без адаптації.
- 🛠 **Підтримуваність:** охайний код легше змінювати й розширювати.
- 🤖 **Автоматизація:** інструменти (лінтери, форматери) стежать за стилем **замість людей** — не треба сперечатись про пробіли на ревʼю.

Далі — від правил (PEP 8, принципи) до інструментів, які їх **автоматично** контролюють.

## 2. 🔎 PEP 8 — офіційний стиль коду Python

**PEP 8** — це загальноприйнятий гайд зі стилю Python (відступи, імена, пробіли). Дотримуватись його — стандарт індустрії.

**Головні правила:**
- **Відступи:** 4 пробіли (не таби).
- **Довжина рядка:** до **79** символів (PEP 8) або **88** (стандарт `black`).
- **Порожні рядки:** 2 — між функціями/класами верхнього рівня, 1 — між методами.
- **Пробіли:** `x = 1` (а не `x=1`), `f(a, b)` (пробіл після коми), без пробілів усередині дужок.
- **Import'и:** кожен з нового рядка, згруповані (стандартні → сторонні → свої).

**Іменування:**

| Що | Стиль | Приклад |
|---|---|---|
| змінні, функції, методи | `snake_case` | `user_name`, `get_total()` |
| класи | `CamelCase` | `UserAccount`, `HttpClient` |
| константи | `UPPER_SNAKE` | `MAX_SIZE`, `PI` |
| «приватне» | `_leading` | `_internal`, `__private` |

Порівняйте два однакові за результатом фрагменти:

In [ ]:
# Той самий код у ДВОХ стилях — результат однаковий, читабельність — ні

# Порушує PEP 8: погані імена, немає пробілів, усе стиснуте
def Calc(X,Y):return X+Y
R=Calc( 2,3 )
print("brudno:", R)

In [ ]:
# За PEP 8: snake_case, пробіли навколо операторів, читабельно
def calc_sum(x, y):
    return x + y


result = calc_sum(2, 3)
print("chysto:", result)

## 3. 🔎 Принципи DRY / KISS / YAGNI

Три короткі принципи, які тримають код простим. **Питають майже завжди.**

- **DRY — Don't Repeat Yourself** («не повторюйся»): будь-яке знання/логіка живе в **одному** місці. Копіпаст → виносьте у функцію/константу. Інакше зміну доведеться робити в 10 місцях (і десь забудете).
- **KISS — Keep It Simple, Stupid** («тримай простим»): найпростіше рішення, що працює. Не ускладнюйте без потреби.
- **YAGNI — You Aren't Gonna Need It** («тобі це не знадобиться»): не пишіть функціонал «про запас, раптом знадобиться». Робіть те, що потрібно **зараз**.

In [ ]:
# DRY: ❌ дублювання проти ✅ винесення повторюваного

# ❌ повторюємо той самий рядок (зміни доведеться робити тричі)
print("Привіт, Аня! Ласкаво просимо.")
print("Привіт, Богдан! Ласкаво просимо.")
print("Привіт, Віка! Ласкаво просимо.")

print("---")

# ✅ DRY: одна функція + цикл — логіка в одному місці
def greet(name):
    print(f"Привіт, {name}! Ласкаво просимо.")

for name in ["Аня", "Богдан", "Віка"]:
    greet(name)

In [ ]:
# KISS: ❌ переускладнено проти ✅ просто (перевірка парності)

# ❌ навіщо так?
def is_even_complicated(n):
    return True if n % 2 == 0 else False

# ✅ простіше нікуди
def is_even(n):
    return n % 2 == 0

print(is_even_complicated(4), is_even(4))

# YAGNI: не додавайте параметри/налаштування "на майбутнє", яких ніхто не просив.
# Напишіть is_even(n). Знадобиться підтримка інших основ числення — додасте ТОДІ.

## 4. 🔎 Type Hinting — анотації типів

**Type hints** підказують, якого **типу** аргументи й що функція **повертає**. Це не змінює роботу програми, зате:
- редактор підказує й ловить помилки ще під час набору;
- код **самодокументований** — видно, що очікується;
- інструмент **mypy** перевіряє типи статично (до запуску).

In [ ]:
# Базовий синтаксис:  ім'я: тип ,  а після  ->  тип, який повертається
def greet(name: str) -> str:
    return f"Привіт, {name}!"

print(greet("Аня"))
print("Анотації:", greet.__annotations__)   # де Python зберігає підказки

In [ ]:
# Анотувати можна й змінні
age: int = 25
price: float = 19.99
is_admin: bool = False
tags: list = ["python", "git"]
print(age, price, is_admin, tags)

In [ ]:
# Складені типи — з модуля typing (працює на всіх версіях Python)
from typing import List, Dict, Optional

def total(prices: List[float]) -> float:
    return sum(prices)

def find_user(uid: int) -> Optional[str]:      # Optional[str] = str АБО None
    users: Dict[int, str] = {1: "Аня", 2: "Богдан"}
    return users.get(uid)

print(total([10.0, 5.5, 2.0]))
print(find_user(1), "/", find_user(99))

**Сучасний коротший синтаксис (Python 3.10+):** замість `Optional[str]` пишуть `str | None`, а замість `List[float]` — `list[float]`. Нижче він працює навіть на 3.9 завдяки рядку `from __future__ import annotations` (робить анотації «відкладеними»).

In [ ]:
from __future__ import annotations   # дозволяє сучасний синтаксис і на Python 3.9

def find_user(uid: int) -> str | None:         # = Optional[str]  (3.10+)
    ...

def total(prices: list[float]) -> float:       # = List[float]    (3.9+)
    return sum(prices)

print("find_user:", find_user.__annotations__)
print("total:   ", total.__annotations__)

In [ ]:
# ВАЖЛИВО: анотації — це лише ПІДКАЗКИ. Python їх НЕ перевіряє під час виконання!
def add(x: int, y: int) -> int:
    return x + y

print(add("a", "b"))   # -> 'ab': передали рядки, і помилки НЕМАЄ (!)
# Таку невідповідність типів упіймає окремий інструмент — mypy (розділ 6).

## 5. 🔎 Docstrings — документація в коді

**Docstring** — рядок у потрійних лапках одразу під `def`/`class`/на початку модуля. Він описує, **що** робить обʼєкт, і доступний через `.__doc__` та `help(obj)`. Стандарт — **PEP 257**.

Популярний **Google-стиль** розділяє `Args` / `Returns` / `Raises`:

In [ ]:
def divide(a: float, b: float) -> float:
    """Ділить a на b.

    Args:
        a: ділене.
        b: дільник (не може бути 0).

    Returns:
        Результат ділення a / b.

    Raises:
        ZeroDivisionError: якщо b дорівнює 0.
    """
    return a / b

print(divide(10, 2))
print("--- docstring ---")
print(divide.__doc__)          # той самий текст покаже й help(divide)

In [ ]:
# Docstring є і в класів, і в методів, і в модулів
class BankAccount:
    """Простий банківський рахунок."""

    def deposit(self, amount: float) -> None:
        """Поповнити рахунок на суму amount."""
        ...

print(BankAccount.__doc__)
print(BankAccount.deposit.__doc__)

> 💡 **Коли писати docstring:** для кожного **публічного** модуля, класу й функції з неочевидною поведінкою. Для очевидного однорядкового гелпера — не обов'язково (не роздувайте шумом). Стилі: **Google**, **NumPy**, **reStructuredText** — оберіть один на проєкт.

## 6. 🔧 Інструменти якості коду

Правила вище контролюють **автоматично** — щоб не робити це руками й не сперечатись на ревʼю. Три різні ролі:

| Інструмент | Роль | Що робить |
|---|---|---|
| **flake8** | лінтер | знаходить стильові й дрібні логічні проблеми (не виправляє) |
| **black** | форматер | автоматично переформатовує код до єдиного стилю |
| **isort** | форматер | сортує та групує `import`'и |
| **mypy** | type checker | статично перевіряє коректність type hints |
| **ruff** | лінтер + форматер | надшвидкий (на Rust); замінює flake8 + isort і форматує як black |

Встановлюються через `pip` (напр. `pip install ruff mypy`).

### Приклади (беремо «брудний» файл `app.py`)
```bash
$ flake8 app.py
app.py:1:1: F401 'os' imported but unused
app.py:3:80: E501 line too long (92 > 79 characters)
app.py:5:1: E302 expected 2 blank lines, found 1
```
```bash
$ black app.py
reformatted app.py
All done! 1 file reformatted.
```
```bash
$ isort app.py
Fixing app.py
```
```bash
$ mypy app.py
app.py:4: error: Argument 1 to "add" has incompatible type "str"; expected "int"  [arg-type]
Found 1 error in 1 file (checked 1 source file)
```

### 🚀 ruff — сучасний дефолт
```bash
$ ruff check app.py         # лінт (дуже швидко)
app.py:1:8: F401 [*] `os` imported but unused
app.py:3:89: E501 Line too long (92 > 88)
Found 2 errors.
[*] 1 fixable with the `--fix` option.

$ ruff check --fix app.py   # авто-виправлення того, що можна
$ ruff format app.py        # форматування (як black)
```

**ruff** настільки швидкий і всеохопний, що в нових проєктах часто **замінює** flake8 + isort + black одним інструментом.

### Лінтер vs форматер vs type checker — у чому різниця

- **Лінтер** (flake8, ruff) — *знаходить* проблеми: невикористаний import, задовгий рядок, підозрілий код. Здебільшого лише **повідомляє**.
- **Форматер** (black, isort, `ruff format`) — *переписує* оформлення коду (пробіли, лапки, перенос рядків) до єдиного стилю. **Не** змінює логіку.
- **Type checker** (mypy) — *перевіряє типи* за вашими анотаціями, ловить `add("a", "b")` ще до запуску.

Разом вони закривають різні аспекти якості й чудово доповнюють одне одного.

### Конфігурація — у `pyproject.toml`
```toml
[tool.ruff]
line-length = 88
lint.select = ["E", "F", "I"]   # E/F — помилки, I — сортування import (як isort)

[tool.mypy]
python_version = "3.12"
strict = true
```

## 7. 🔎 pre-commit hooks — автоперевірка перед комітом

Домовитись «усі запускають лінтери» — недостатньо: хтось забуде. **Git hooks** — це скрипти, які Git запускає на певні події. **pre-commit** — популярний інструмент, що на кожен `git commit` **автоматично** проганяє лінтери/форматери й **не дасть закомітити**, поки код брудний.

```
git commit ──► pre-commit проганяє ruff, mypy, ... ──► ✅ усе ок ──► коміт створено
                        │
                        └─► ❌ знайдено проблему ──► коміт ЗАБЛОКОВАНО
                                                     (виправ і закоміть знову)
```

**Налаштування (3 кроки):**
1. Створити файл `.pre-commit-config.yaml` у корені репозиторію:
```yaml
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.6.0
    hooks:
      - id: ruff            # лінт
        args: [--fix]
      - id: ruff-format     # форматування
  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.11.0
    hooks:
      - id: mypy            # перевірка типів
```
2. Встановити й «увімкнути» hook у репозиторії:
```bash
pip install pre-commit
pre-commit install        # тепер hooks спрацьовують на кожен git commit
```
3. (За бажанням) прогнати на всіх файлах одразу:
```bash
pre-commit run --all-files
```

> ✅ **Ефект:** у репозиторій фізично **не потрапляє** код, що не проходить лінтер/типи. Стиль стає єдиним для всієї команди автоматично. Ті самі перевірки зазвичай дублюють ще й у **CI** (GitHub Actions) — на випадок, якщо хтось вимкнув хук локально.

## 8. Рекомендований стек (сучасний, простий)

Для нового проєкту достатньо **двох** інструментів + pre-commit:

- **ruff** — лінт + форматування (замість flake8 + isort + black);
- **mypy** — перевірка типів;
- **pre-commit** — щоб це запускалось саме, перед кожним комітом.

```bash
pip install ruff mypy pre-commit
pre-commit install
```

Плюс мінімальний `pyproject.toml` (розділ 6) — і ваш проєкт має професійну гігієну коду з першого дня. Саме такий підхід використовують у скелеті проєкту з Модуля 2.

# ✅ Підсумок уроку

- **Якість коду** = читабельність + послідовність; код читають частіше, ніж пишуть.
- **PEP 8** — стиль Python: 4 пробіли, `snake_case`/`CamelCase`/`UPPER_SNAKE`, пробіли, порядок import'ів.
- **DRY** (не повторюйся), **KISS** (тримай простим), **YAGNI** (не пиши про запас).
- **Type hints** — `name: str -> str`; підказки, які **не** перевіряються в рантаймі (це робить **mypy**). Складені типи — `typing` або сучасний `str | None`.
- **Docstrings** — `"""..."""` під `def`/`class`; доступні через `__doc__`/`help()`; стилі Google/NumPy (PEP 257).
- **Інструменти:** **flake8** (лінтер), **black**/**isort** (форматери), **mypy** (типи), **ruff** (усе разом, швидко).
- **pre-commit hooks** — автоперевірка перед кожним `git commit`; брудний код не потрапляє в репозиторій.

### 📝 Домашнє завдання
[domashnie-zavdannia-lektsiya-2.ipynb](domashnie-zavdannia-lektsiya-2.ipynb)

### ▶️ Далі
Ви завершили Модуль 4. Тепер у вас є повний базовий інструментарій розробника: Python, ООП, БД,
Git і культура якості коду — з цим уже беруться за реальні проєкти й проходять співбесіди.

### 📚 Хочу знати більше
- PEP 8 (офіційно): <https://peps.python.org/pep-0008/>
- PEP 257 (docstrings): <https://peps.python.org/pep-0257/>
- Документація ruff: <https://docs.astral.sh/ruff/>
- mypy (type checking): <https://mypy.readthedocs.io/>
- pre-commit: <https://pre-commit.com/>
- typing (модуль): <https://docs.python.org/3/library/typing.html>